# Offline ML Risk Prioritization for Test Backlogs

Here we simulate reading a large CSV of hundreds of User Stories and prioritizing them using our pre-trained RandomForest model.

In [ ]:
import pandas as pd
import joblib

# Load Model & Encoder
clf = joblib.load('../data/risk_model.pkl')
le = joblib.load('../data/label_encoder.pkl')

def extract_features(text):
    text = str(text).lower()
    return {
        'loc': len(text) // 10,
        'cyclomatic_complexity': len(text.split()) // 5 + 5,
        'prev_defects': 4 if any(w in text for w in ['transfer', 'payment', 'money', 'withdraw']) else 1,
        'negative_tests': 0,
        'edge_tests': 0,
        'money_related': 1 if any(w in text for w in ['amount', 'money', 'transfer', 'currency', '$']) else 0,
        'security_related': 1 if any(w in text for w in ['2fa', 'password', 'auth', 'security', 'token', 'otp']) else 0
    }


In [ ]:
# Simulate a massive backlog (e.g. from Jira export)
backlog_data = [
    {"id": "US-101", "story": "As a user, I want to see my profile picture so I know I am logged in."},
    {"id": "US-102", "story": "As a user, I want to transfer money international with 2FA."},
    {"id": "US-103", "story": "As an admin, I want to suspend accounts."}
]
df = pd.DataFrame(backlog_data)

# Score them
scores = []
for idx, row in df.iterrows():
    feats = extract_features(row['story'])
    X_pred = pd.DataFrame([feats])
    risk_prob = max(clf.predict_proba(X_pred)[0]) * 100
    scores.append(round(risk_prob, 1))

df['Risk Score'] = scores
df_prioritized = df.sort_values(by='Risk Score', ascending=False)
print("Top 5 Riskiest Stories to Automate First:")
display(df_prioritized.head())
